# Nghiên cứu Ngưỡng Cảnh báo Nước Xâm nhập & Môi trường Hầm hàng (SVMG)

> **Lưu ý phạm vi:** Đây là nguyên mẫu hỗ trợ giám sát dùng cho nghiên cứu/đào tạo, **không** phải thiết bị an toàn hàng hải được chứng nhận SOLAS/IMO. Mọi nguồn khoa học và tiêu chuẩn hàng hải liệt kê dưới đây **phải được kiểm chứng độc lập** trước khi xác nhận làm ngưỡng vận hành chính thức.

## 1. Vì sao chọn phương án Cấu hình Động thay vì Dùng Hai Cảm biến Nước Cố định?

Trong thiết kế hệ thống giám sát nước xâm nhập cho Node 2 (Cargo Node), nhóm cân nhắc 2 phương án:
1. **Phương án hai cảm biến mức cố định (2-point float / level switches):** Lắp 2 cảm biến tại độ cao cố định (ví dụ 500 mm và 2000 mm).
2. **Phương án cấu hình động thông số hầm hàng (Dynamic Hold Configuration):** Giữ 1 cảm biến mức liên tục (chiết áp / cảm biến áp suất thủy tĩnh quy đổi 0–100%), kết hợp cấu hình động qua `PUT /api/v1/devices/{cargo_id}/cargo/config` gồm:
   - `cargo_hold_depth_mm`: Chiều sâu thực tế của hầm hàng (mm).
   - `water_sensor_full_scale_mm`: Thang đo tối đa tương ứng 100% của cảm biến nước (mm).
   - `vessel_type`, `cargo_type`: Loại tàu và loại hàng hóa.
   - `sensor_calibrated`: Cờ xác nhận cảm biến thực tế đã được hiệu chuẩn có căn cứ.

### Lý do lựa chọn phương án Cấu hình Động
- **Không phụ thuộc kích thước cố định của một tàu duy nhất:** Mỗi tàu và mỗi hầm hàng có chiều sâu $D_{\text{hold}}$ khác nhau (ví dụ tàu ven biển 4.500 mm so với tàu hàng rời Panamax 15.000 mm). Nếu dùng 2 cảm biến gắn cứng hoặc ngưỡng `%` cố định (như 35%/70%), cùng một mức nước 500 mm có thể tương ứng các tỷ lệ hoàn toàn khác nhau trên hai tàu.
- **Tách biệt phần cứng và logic nghiệp vụ:** Khi thay đổi loại tàu, loại hàng hóa hoặc lắp cảm biến có thang đo khác nhau, kỹ sư/Supervisor chỉ cần cập nhật cấu hình qua API/UI mà không phải sửa lại mạch phần cứng hay nạp lại firmware ESP32.
- **Đo liên tục theo đơn vị vật lý (mm):** Backend quy đổi trực tiếp $h_{\text{mm}} = \text{pct} \times \frac{S_{\text{full}}}{100}$ và tính ngưỡng nguy cấp theo chiều sâu hầm $C_{\text{water}} = \min(0{,}15 \times D_{\text{hold}}, 2000\text{ mm})$.
- **An toàn khi thiếu thông số:** Khi chưa khai báo đủ `cargo_hold_depth_mm` và `water_sensor_full_scale_mm`, hệ thống tuyệt đối **không tự giả định kích thước hầm**, mà giữ trạng thái `UNKNOWN (NOT_CONFIGURED)` và hướng dẫn người vận hành bổ sung.

## 2. Nguồn Tham chiếu & Yêu cầu Kiểm chứng Độc lập

| Chỉ số | Đơn vị | Ngưỡng mặc định (Prototype) | Nguồn tham chiếu được ghi nhận | Ghi chú kiểm chứng |
|---|---|---|---|---|
| Nước xâm nhập (`WATER`) | `mm` | `WARNING ≥ 500 mm`; `CRITICAL ≥ min(0.15 × D_hold, 2000 mm)` | SOLAS Chapter XII Reg. 12; IMO Resolution MSC.188(79)/Rev.2 | Tham chiếu cho hệ thống phát hiện ngập nước hầm tàu hàng rời; **cần đối chiếu bản văn gốc IMO/SOLAS** cho từng hạng tàu cụ thể. |
| Nhiệt độ không khí hầm (`TEMPERATURE`) | `°C` | `WARNING ≥ 35°C`; `CRITICAL ≥ 40°C` | Wang & Zhou (2023), *Applied Sciences*, DOI: `10.3390/app132312878` | DHT22 đo nhiệt độ **không khí trong hầm** (`AIR_IN_HOLD`), không phải nhiệt độ lõi khối hàng. |
| Độ ẩm không khí hầm (`HUMIDITY`) | `%RH` | `WARNING ≥ 75%`; `CRITICAL ≥ 90%` | IMO/ILO/UNECE CTU Code (2014); *Food Science & Nutrition* (2024), DOI: `10.1002/fsn3.4302` | Ngưỡng kỹ thuật prototype cho hàng nông sản khô; **phải kiểm chứng độc lập** theo loại hàng thực tế trước khi áp dụng. |

> **Phân biệt nguồn dữ liệu:** Hệ thống tách biệt rõ `SIMULATION` (dữ liệu mô phỏng), `SENSOR_UNCALIBRATED` (cảm biến thực tế chưa hiệu chuẩn), và `SENSOR_CALIBRATED` (cảm biến thực tế đã có căn cứ hiệu chuẩn).

In [ ]:
# Minh họa công thức tính ngưỡng nước xâm nhập động cho nhiều kích thước hầm hàng khác nhau
def compute_water_thresholds(depth_mm: float, warning_mm: float = 500.0, fraction: float = 0.15, cap_mm: float = 2000.0):
    critical_mm = min(fraction * depth_mm, cap_mm)
    if critical_mm <= warning_mm:
        raise ValueError(f'Chiều sâu {depth_mm} mm quá nhỏ: CRITICAL ({critical_mm} mm) <= WARNING ({warning_mm} mm)')
    return {'depth_mm': depth_mm, 'warning_mm': warning_mm, 'critical_mm': critical_mm}

for d in [4500, 8000, 10000, 15000, 20000]:
    print(compute_water_thresholds(d))